# Code Attribution

Portions of this code are adapted from:

**FLASH: A Comprehensive Approach to Intrusion Detection via Provenance Graph Representation Learning**

Source: [https://github.com/DART-Laboratory/Flash-IDS](https://github.com/DART-Laboratory/Flash-IDS)


**threaTrace: Detecting and Tracing Host-based Threats in Node Level Through Provenance Graph Learning**

Source: [https://github.com/threaTrace-detector/threaTrace](https://github.com/threaTrace-detector/threaTrace)

## Importing required packages

In [ ]:
import os
from tqdm import tqdm
import torch
import torch.nn.functional as F
import pandas as pd
from collections import defaultdict
import numpy as np
import re
import gdown
import requests
import pickle
from torch_geometric.nn import SAGEConv
from sklearn.metrics.pairwise import cosine_similarity
from utils_trace.data_process_train import *
from utils_trace.data_process_test import *
from utils_trace.evaluate_darpatc import *

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Model Class definition

In [2]:
class SAGENet(torch.nn.Module):
	def __init__(self, in_channels, out_channels):
		super(SAGENet, self).__init__()	
		self.conv1 = SAGEConv(in_channels, 32, normalize=False)
		self.conv2 = SAGEConv(32, out_channels, normalize=False)

	def forward(self, x, edge_index):
		x = self.conv1(x, edge_index)
		x = F.relu(x)
		x = F.dropout(x, training=self.training)
		x = self.conv2(x, edge_index)
		return F.log_softmax(x, dim=1) 

## Download Dataset and alarm/Ground Truth files

In [3]:
FOLDER_URL = "https://drive.google.com/drive/folders/1m6dg0KH6028iOXOX61Vrxkkd8p-fhLAX"

output_dir = os.path.abspath("../groundtruth/trace")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=True
)

Retrieving folder contents


Processing file 1oIhRhArvjEc3-xKgU9x21d-JY-ByXYZC alarm_evasion.txt
Processing file 1C1IvG6URHNtzeg6YsLSAuNoMKGo4bvo- alarm.txt
Processing file 1r1LrdpYgCGyg9m_BaAq3ukrYd5-iGma0 groundtruth_nodeId.txt
Processing file 1pf5edjBl-HOsDcz8ELLankrrbAH0_Evn groundtruth_uuid.txt
Processing file 1zXRWoPR1E8KwfvHWve9s3LFzV_WDVhqt id_to_uuid.txt


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1oIhRhArvjEc3-xKgU9x21d-JY-ByXYZC
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/groundtruth/trace/alarm_evasion.txt
100%|██████████| 22.1M/22.1M [00:00<00:00, 47.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1C1IvG6URHNtzeg6YsLSAuNoMKGo4bvo-
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/groundtruth/trace/alarm.txt
100%|██████████| 23.1M/23.1M [00:00<00:00, 65.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1r1LrdpYgCGyg9m_BaAq3ukrYd5-iGma0
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/groundtruth/trace/groundtruth_nodeId.txt
100%|██████████| 3.91M/3.91M [00:00<00:00, 21.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1pf5edjBl-HOsDcz8ELLankrrbAH0_Evn
To: /

['/home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/groundtruth/trace/alarm_evasion.txt',
 '/home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/groundtruth/trace/alarm.txt',
 '/home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/groundtruth/trace/groundtruth_nodeId.txt',
 '/home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/groundtruth/trace/groundtruth_uuid.txt',
 '/home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/groundtruth/trace/id_to_uuid.txt']

In [4]:
os.makedirs("trace_resources", exist_ok=True)

file_ids = [
    "1Dxmr0vXCJYUKnqeLZTgiv9OgEEvVUnGG",
    "1VcR0J-wfbLFvIgiK8W5y6ZpBbnnAcZIN"
]

def get_filename(file_id):
    url = f"https://drive.google.com/file/d/{file_id}/view"
    html = requests.get(url).text
    match = re.search(r'<meta property="og:title" content="(.+?)"', html)
    return match.group(1) if match else file_id

for fid in file_ids:
    filename = get_filename(fid)
    out_path = os.path.join("trace_resources", filename)
    url = f"https://drive.google.com/uc?id={fid}"
    
    print(f"⬇️ Downloading: {filename}")
    gdown.download(url, out_path, quiet=False)

⬇️ Downloading: for_trace.pkl


Downloading...
From (original): https://drive.google.com/uc?id=1Dxmr0vXCJYUKnqeLZTgiv9OgEEvVUnGG
From (redirected): https://drive.google.com/uc?id=1Dxmr0vXCJYUKnqeLZTgiv9OgEEvVUnGG&confirm=t&uuid=56c9529d-9dd4-45a2-87cc-831307affe81
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/TRACE/trace_resources/for_trace.pkl
100%|██████████| 630M/630M [00:06<00:00, 92.0MB/s] 


⬇️ Downloading: trace_test.txt


Downloading...
From (original): https://drive.google.com/uc?id=1VcR0J-wfbLFvIgiK8W5y6ZpBbnnAcZIN
From (redirected): https://drive.google.com/uc?id=1VcR0J-wfbLFvIgiK8W5y6ZpBbnnAcZIN&confirm=t&uuid=b928452c-ad51-4799-a2d2-80b4013c3de5
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/TRACE/trace_resources/trace_test.txt
100%|██████████| 467M/467M [00:04<00:00, 95.9MB/s] 


## Download the models and features

In [5]:
FOLDER_URL = "https://drive.google.com/drive/folders/1emRGLLwVObkM7Gp-1rpl0hzGLZbMdHpJ"

output_dir = os.path.abspath("../models/trace")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

print(f"All files downloaded into: {output_dir}")

Retrieving folder contents


Processing file 1fzuUoM5dfMYoBOpr90HAA_VRbLbAEn66 feature.txt
Processing file 1-Xa3pZCbTeo7Pgqx1g3hI46ucif067ov label.txt
Processing file 1qeq_hhojeKhGUpvBmnEgmJQa_pUHpH36 model_0
Processing file 1HhjZ5MmGOjeIUW7qy1CK8urStE8Xgbxe model_1
Processing file 1N1_2kxTsRHROloYEzJDlYnpymg9Q3KjI model_2
Processing file 1sk8Ydnqb1P_ZMfRQcYfSGhbzcrAUka4C model_3
Processing file 1trF2QvMg_1sHUW6rCx8un7dc2oUjHJGv model_4
Processing file 1jLyf1H3SO6m1jdhjCHVbFgHdAKNIS3i3 model_5
Processing file 15Pa7jWTc2-QftioWQn0dhFUatMH4GH1E model_6
Processing file 1RGzajuij3qzEwY2rK60uvixaZRZOjKvI model_7
Processing file 1z3Anu-cehQTTBjpQHtRojdsRHpII6WcR model_8
Processing file 1Wngw0VaS0Sx53y805L42PxHJChamKM6L model_9
Processing file 1buZHs0JDk5uIvsUy06CHRSDx6kc0YWEB model_10
Processing file 1mg8M3j-JywgAoHo8BwwwMgf8ET-vCUXK model_11
Processing file 17bIM3L-vYXvmuYlKVv3nC61BWlm19Pgm model_12
Processing file 1Vee3b0GHC5tNNwzzwjUFlTRfoVqn4g0_ model_13
Processing file 1uODGOccoy-UfWFiBwXq3wGdJDOd4MaK- model_14


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1fzuUoM5dfMYoBOpr90HAA_VRbLbAEn66
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/models/trace/feature.txt
100%|██████████| 387/387 [00:00<00:00, 3.34MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-Xa3pZCbTeo7Pgqx1g3hI46ucif067ov
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/models/trace/label.txt
100%|██████████| 204/204 [00:00<00:00, 1.42MB/s]
Downloading...
From: https://drive.google.com/uc?id=1qeq_hhojeKhGUpvBmnEgmJQa_pUHpH36
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/models/trace/model_0
100%|██████████| 16.8k/16.8k [00:00<00:00, 1.98MB/s]
Downloading...
From: https://drive.google.com/uc?id=1HhjZ5MmGOjeIUW7qy1CK8urStE8Xgbxe
To: /home/student/rastogiv/Desktop/Evasive-ToPubl

All files downloaded into: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/models/trace



Download completed


In [6]:
FOLDER_URL = "https://drive.google.com/drive/folders/1M0ZTCIOoy_Si5xTmXU7csX8OKPqlEzNe"

output_dir = os.path.abspath("../models/trace")
os.makedirs(output_dir, exist_ok=True)

gdown.download_folder(
    url=FOLDER_URL,
    output=output_dir,
    quiet=False,
    use_cookies=False
)

print(f"All files downloaded into: {output_dir}")

Retrieving folder contents


Processing file 1z-PDbSRktUF5bBfXaPx-bAKgXKs0BepZ best_refined_model.pkl
Processing file 1wEazc7WWGnopdqoahJtjp6kmJsYgO3fW trace_model_weights.pth


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1z-PDbSRktUF5bBfXaPx-bAKgXKs0BepZ
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/models/trace/best_refined_model.pkl
100%|██████████| 20.1k/20.1k [00:00<00:00, 951kB/s]
Downloading...
From: https://drive.google.com/uc?id=1wEazc7WWGnopdqoahJtjp6kmJsYgO3fW
To: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/models/trace/trace_model_weights.pth
100%|██████████| 16.7k/16.7k [00:00<00:00, 1.83MB/s]

All files downloaded into: /home/student/rastogiv/Desktop/Evasive-ToPublish/EvasiveAttack/CODE_TO_PUBLISH/ThreaTrace/models/trace



Download completed


## Load test data

In [7]:
with open("trace_resources/for_trace.pkl", "rb") as file:
    df, nodes, phrases, labels, edges, mapp, GT_mal, all_ids, indices_of_malicious_nodes = pickle.load(file)

# --- Load test dataset ---
path = 'trace_resources/trace_test.txt' 
data1, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'trace')
dataset_test = TestDatasetA(data1)
data_test = dataset_test[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [8]:
label_num

11

## Function that evaluates the model and returns metrics

In [9]:
def test_one_model(mask, model, device, data, thre, nodeA):
    model.eval()
    data = data.to(device)

    # start with all nodes flagged as suspicious
    updated_mask = torch.ones(data.num_nodes, dtype=torch.bool, device=device)

    # Convert malicious list to set for O(1) lookup
    malicious_set = set(nodeA)

    with torch.no_grad():
        out = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        pro = torch.softmax(out, dim=1)
        pro1 = pro.max(1)

        # suppress top prediction to find 2nd-best for thresholding
        pro_clone = pro.clone()
        pro_clone[torch.arange(len(pro)), pro1[1]] = -1
        pro2 = pro_clone.max(1)

        # apply threshold logic
        uncertain_mask = (pro1[0] / pro2[0]) < thre
        pred[uncertain_mask] = 100  # dummy label for uncertain predictions

        # evaluation
        TP, FP, TN, FN = 0, 0, 0, 0
        test_nodes = mask.nonzero(as_tuple=False).view(-1).tolist()

        for i in tqdm(test_nodes, desc="Evaluating nodes"):
            flagged = (pred[i] != data.y[i])  # mismatch = flagged
            
            if not flagged:
                updated_mask[i] = False   # node is cleared

            if i in malicious_set:  # malicious node
                if flagged:
                    TP += 1
                else:
                    FN += 1
            else:  # benign node
                if flagged:
                    FP += 1
                else:
                    TN += 1

        # metrics
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = FP / (FP + TN) if (FP + TN) > 0 else 0.0

        acc = (pred[mask] == data.y[mask]).sum().item() / mask.sum().item()

    return acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask

## Evaluating before evasion

In [10]:
thre = 1

model_good = SAGENet(feature_num, label_num).to(device)

print(feature_num, label_num)

load_path = "../models/trace/trace_model_weights.pth"
model_good.load_state_dict(torch.load(load_path, map_location=device))

acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask = test_one_model(data_test.test_mask, model_good, device, data_test, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

46 11


Evaluating nodes: 100%|██████████| 1211448/1211448 [00:22<00:00, 53511.74it/s]

Test Accuracy: 0.7838
TP: 67380, FP: 194499, TN: 949566, FN: 3
Precision: 0.2573, Recall: 1.0000, F1: 0.4093, FPR: 0.1700


## Functions to prepare the Graph

In [11]:
def add_node_properties(nodes, node_id, properties):
    if node_id not in nodes:
        nodes[node_id] = []
    nodes[node_id].extend(properties)

def update_edge_index(edges, edge_index, index):
    for src_id, dst_id in edges:
        src = index[src_id]
        dst = index[dst_id]
        edge_index[0].append(src)
        edge_index[1].append(dst)

def prepare_graph(df):
    nodes, labels, edges = {}, {}, []
    dummies = {'SUBJECT_PROCESS': 0, 'FILE_OBJECT_FILE': 1, 'FILE_OBJECT_UNIX_SOCKET': 2, 
               'UnnamedPipeObject': 3, 'NetFlowObject': 4, 'FILE_OBJECT_DIR': 5}
    
    for _, row in df.iterrows():
        action = row["action"]
        properties = [row['exec'], action] + ([row['path']] if row['path'] else [])
        
        actor_id = row["actorID"]
        add_node_properties(nodes, actor_id, properties)
        labels[actor_id] = dummies[row['actor_type']]

        object_id = row["objectID"]
        add_node_properties(nodes, object_id, properties)
        labels[object_id] = dummies[row['object']]

        edges.append((actor_id, object_id))

    features, feat_labels, edge_index, index_map = [], [], [[], []], {}
    for node_id, props in nodes.items():
        features.append(props)
        feat_labels.append(labels[node_id])
        index_map[node_id] = len(features) - 1

    update_edge_index(edges, edge_index, index_map)

    return features, feat_labels, edge_index, list(index_map.keys())

## Evasion

In [12]:
with open("trace_resources/for_trace.pkl", "rb") as file:
    df, nodes, phrases, labels, edges, mapp, GT_mal, all_ids, indices_of_malicious_nodes = pickle.load(file)

# --- Load test dataset ---
path = 'trace_resources/trace_test.txt' 
data1, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'trace')
dataset_test = TestDatasetA(data1)
data_test = dataset_test[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


## Step 1: Benign Node Selection

In [13]:
def split_malicious_nodes_by_label(malicious_indices, labels):

    r"""Groups malicious nodes based on their labels.

    :param malicious_indices: List of indices for malicious nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of malicious node indices as values.
    """

    malicious_groups = defaultdict(list)

    for node_idx in malicious_indices:
        node_label = labels[node_idx]
        malicious_groups[node_label].append(node_idx)

    return dict(malicious_groups)

malicious_groups = split_malicious_nodes_by_label(indices_of_malicious_nodes, labels)

print("The number of malicious nodes for each label is as follows:")

for label, nodes in malicious_groups.items():
    print(f"Label {label}: {len(nodes)}")  

The number of malicious nodes for each label is as follows:
Label 10: 67339
Label 1: 18
Label 0: 8
Label 6: 6
Label 3: 10
Label 2: 1
Label 4: 1


In [14]:
def split_nodes_by_label(nodes, labels):

    r"""Groups all nodes based on their labels.

    :param nodes: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :return: Dictionary with label as key and list of node indices as values.
    """

    node_groups = defaultdict(list)
    for node_idx, _ in enumerate(nodes):
        node_label = int(labels[node_idx].item()) 
        node_groups[node_label].append(node_idx)
    return dict(node_groups)

all_nodes_grouped = split_nodes_by_label(data_test.x, data_test.y)

print("The number of nodes (either malicious or benign) for each label is as follows:")
for label, nodes in sorted(all_nodes_grouped.items()):
    print(f"Label {label}: {len(nodes)}")


The number of nodes (either malicious or benign) for each label is as follows:
Label 0: 5151
Label 1: 590122
Label 2: 10
Label 3: 94376
Label 4: 11772
Label 5: 147226
Label 6: 28
Label 7: 527
Label 8: 1163
Label 9: 44
Label 10: 361029


In [15]:
def split_nodes_by_label_exclude_malicious(nodes, labels, indices_of_malicious_nodes):

    r"""Groups benign (non-malicious) nodes based on their labels.

    :param nodes: List of phrases (interactions) for all nodes.
    :param labels: List of labels for all nodes.
    :param indices_of_malicious_nodes: List of indices for malicious nodes.
    :return: Dictionary with label as key and list of benign node indices as values.
    """

    node_groups = defaultdict(list)
    malicious_set = set(indices_of_malicious_nodes)  

    for node_idx, _ in enumerate(nodes):
        if node_idx not in malicious_set:
            node_label = int(labels[node_idx].item()) 
            node_groups[node_label].append(node_idx)

    return dict(node_groups)


all_nodes_grouped_excluding_malicious = split_nodes_by_label_exclude_malicious(
    data_test.x, data_test.y, indices_of_malicious_nodes
)

print("The number of candidate benign nodes for each label is as follows:")
for label, nodes in sorted(all_nodes_grouped_excluding_malicious.items()):
    print(f"Label {label}: {len(nodes)}")


The number of candidate benign nodes for each label is as follows:
Label 0: 5148
Label 1: 590085
Label 2: 10
Label 3: 27064
Label 4: 11771
Label 5: 147198
Label 6: 28
Label 7: 527
Label 8: 1161
Label 9: 44
Label 10: 361029


## Step 2: Minimal Interaction Selection

In [16]:
def filter_nodes_by_phrase_length(node_groups, phrases, min_len=5, max_len=10): 

    r"""Filters the benign nodes in each label based on the number of interactions (phrase length).

    :param node_groups: Dictionary with label as key and list of benign node indices as values.
    :param phrases: List of phrases (interactions) for all nodes.
    :param min_len: Minimum number of interactions (hyperparameter).
    :param max_len: Maximum number of interactions (hyperparameter).
    :return: Dictionary with label as key and list of filtered benign node indices as values.
    """

    filtered_groups = {}

    for label, node_indices in node_groups.items():
        filtered_groups[label] = [
            idx for idx in node_indices if min_len <= len(phrases[idx]) <= max_len
        ]

    return filtered_groups


filtered_nodes_by_interactions = filter_nodes_by_phrase_length(all_nodes_grouped_excluding_malicious, phrases)

print("The number of candidate benign nodes for each label is as follows:")

for label, nodes in filtered_nodes_by_interactions.items():
    print(f"Label {label}: {len(nodes)}")  

The number of candidate benign nodes for each label is as follows:
Label 0: 343
Label 10: 44374
Label 1: 22354
Label 8: 47
Label 3: 846
Label 4: 2272
Label 5: 15368
Label 2: 1
Label 7: 131
Label 6: 6
Label 9: 0


## Step 3: Contextual Consistency Filtering

In [17]:
def compute_top_similar_nodes(malicious_groups, filtered_nodes_grouped, nodes):

    r"""Orders the benign nodes within each label based on their contextual similarity to the malicious nodes in the same label in descending order.

    :param malicious_groups: Dictionary with label as key and list of malicious node indices as values.
    :param filtered_nodes_grouped: Dictionary with label as key and list of filtered benign node indices as values.
    :param nodes: Feature vectors (embeddings) for all nodes.
    :return: Dictionary with label as key and list of sorted benign node indices as values.
    """

    top_nodes_by_similarity = {}

    for label in tqdm(malicious_groups.keys()):
        if label in filtered_nodes_grouped and len(filtered_nodes_grouped[label]) > 0:
            malicious_indices = malicious_groups[label]
            filtered_indices = filtered_nodes_grouped[label]

            # Get feature vectors for malicious and filtered nodes
            malicious_vectors = np.array([nodes[idx] for idx in malicious_indices])
            filtered_vectors = np.array([nodes[idx] for idx in filtered_indices])

            # Compute cosine similarity (malicious nodes vs filtered nodes)
            similarities = cosine_similarity(filtered_vectors, malicious_vectors)  # Shape: (filtered, malicious)

            # Take the max similarity score for each filtered node
            max_similarities = similarities.max(axis=1)  # Max similarity per filtered node

            # Use np.argsort to get the indices that would sort the max_similarities array
            sorted_indices = np.array(filtered_indices)[np.argsort(-max_similarities)]  # Negative for descending order

            # Select top most similar nodes for the current label
            top_nodes_by_similarity[label] = sorted_indices

    return top_nodes_by_similarity

top_nodes_by_similarity = compute_top_similar_nodes(malicious_groups, filtered_nodes_by_interactions, data_test.x.cpu().numpy())

print("The number of candidate benign nodes for each label is as follows:")

for label, top_nodes in top_nodes_by_similarity.items():
    print(f"Label {label}: {len(top_nodes)}") 

100%|██████████| 7/7 [00:08<00:00,  1.21s/it]

The number of candidate benign nodes for each label is as follows:
Label 10: 44374
Label 1: 22354
Label 0: 343
Label 6: 6
Label 3: 846
Label 2: 1
Label 4: 2272


## Graph Structure Adjustment

In [18]:
def graph_modification(df, indices_of_malicious_nodes, best_benign_replacements, mapp, labels):

    r"""
    Modifies the graph by adding edges between malicious and selected benign nodes.
    :param df: Original DataFrame containing graph edges.
    :param indices_of_malicious_nodes: List of indices for malicious nodes.
    :param best_benign_replacements: Dictionary with label as key and the most confident benign node index as value.
    :param mapp: List mapping node indices to node IDs.
    :param labels: List of labels for all nodes.
    :return: Modified DataFrame with new edges added.
    """
    # Prepare the DataFrame for modification
    print("Number of rows in df before modification:", len(df))

    # Step 1: Create a mapping of malicious node ID → most similar benign node ID

    malicious_to_benign = {}
    for i in nodeA:
        if labels[i] in top_nodes_by_similarity and len(top_nodes_by_similarity[labels[i]]) > 0:
            malicious_to_benign[mapp[i]] = mapp[top_nodes_by_similarity[labels[0]][0]]

    # Step 2: Retrieve all benign node IDs that will be replaced
    all_benign_replacements = set(malicious_to_benign.values())

    # Step 3: Extract relevant rows for each benign ID
    benign_rows_dict = {benign_id: df[(df['actorID'] == benign_id) | (df['objectID'] == benign_id)].copy()
                        for benign_id in all_benign_replacements}

    new_rows = []

    for malicious_id, benign_id in tqdm(malicious_to_benign.items(), desc="Processing malicious nodes"):
        if benign_id in benign_rows_dict:
            modified_rows = benign_rows_dict[benign_id].copy()  # Copy rows where benign ID appears
            modified_rows.loc[modified_rows['actorID'] == benign_id, 'actorID'] = malicious_id
            modified_rows.loc[modified_rows['objectID'] == benign_id, 'objectID'] = malicious_id
            new_rows.append(modified_rows)

    df_curated = pd.concat([df] + new_rows, ignore_index=True)

    print("Number of rows in df after modification:", len(df_curated))
    print("The number of triggered events are: ", len(df_curated) - len(df))

    return df_curated

df_curated = graph_modification(df, indices_of_malicious_nodes, top_nodes_by_similarity, mapp, labels)


Number of rows in df before modification: 2255113


Processing malicious nodes: 100%|██████████| 67367/67367 [00:12<00:00, 5357.82it/s]


Number of rows in df after modification: 2524581
The number of triggered events are:  269468


In [19]:
# Save the curated DataFrame to a text file (tab-separated, no header, no index)
output_path = "trace_resources/trace_test_curated.txt"
df_curated.to_csv(output_path, sep='\t', index=False, header=False)

print(f"Curated file saved as {output_path}")


Curated file saved as trace_resources/trace_test_curated.txt


## Evaluation after evasion

In [20]:
# Load and process test dataset
path = 'trace_resources/trace_test_curated.txt' 
data_curated, feature_num, label_num, adj, adj2, nodeA = MyDatasetA(path, 'trace')
dataset_curated = TestDatasetA(data_curated)
data_curated = dataset_curated[0] 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [21]:
# Load the whole model (architecture + weights)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

load_path = "../models/trace/trace_model_weights.pth"
model_good = SAGENet(feature_num, label_num).to(device)
model_good.load_state_dict(torch.load(load_path, map_location=device))

# Put in eval mode for inference
model_good.eval()
thre = 1

print("Model loaded successfully:", type(model_good))

acc, TP, FP, TN, FN, precision, recall, f1, fpr, updated_mask_evasion = test_one_model(data_curated.test_mask, model_good, device, data_curated, thre, nodeA)

print(f"Test Accuracy: {acc:.4f}")
print(f"TP: {TP}, FP: {FP}, TN: {TN}, FN: {FN}")
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, FPR: {fpr:.4f}")

Model loaded successfully: <class '__main__.SAGENet'>


Evaluating nodes: 100%|██████████| 1211448/1211448 [00:23<00:00, 52136.39it/s]

Test Accuracy: 0.8576
TP: 73, FP: 172470, TN: 971595, FN: 67310
Precision: 0.0004, Recall: 0.0011, F1: 0.0006, FPR: 0.1508
